(r-comparisons-intertemporal)=

# Intertemporal treatment in ModernDiD and R

This comparison checks whether {func}`~moderndid.did_multiplegt` and R's
`DIDmultiplegtDYN::did_multiplegt_dyn` return the same effects for the
banking deregulation panel. Both estimate two post-treatment horizons and
one placebo without normalization or covariates and cluster analytical
standard errors by state. The output shows each fitted result before
comparing the estimates and standard errors directly.

R runs inside this Python session through `rpy2` and receives the same
data as the Python estimator. See the section on
{ref}`running the comparisons <r-comparisons-running>` for the setup
that runs these cells locally.

In [1]:
import numpy as np
import pandas as pd
import polars as pl
from rpy2 import robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

import moderndid as did

pd.set_option("display.precision", 8)

# DIDmultiplegtDYN 2.4.0 uses the attached R polars package.
with ro.default_converter.context():
    r_polars = importr("polars")
    r_did = importr("DIDmultiplegtDYN")

Error importing in API mode: ImportError("dlopen(/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <05EE76C0-33BF-3D4C-BC56-84005241BC88> /Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


## The same county panel

{func}`~moderndid.load_favara_imbs` loads the county panel used in the
{ref}`intertemporal treatment example <example_inter_did>`.
`Dl_vloans_b` measures the change in log mortgage lending. `inter_bra`
counts the banking restrictions that a county's state had lifted in each
year and can change more than once. The 50 state codes in `state_n` define
the clusters in both calls.

We keep the 928 of 1,048 counties whose lending outcome is observed in all
12 years from 1994 through 2005. Since both estimators receive this complete
panel of 11,136 county-years, the comparison does not depend on how either
implementation handles incomplete county histories.

In [2]:
data = did.load_favara_imbs().select("county", "year", "Dl_vloans_b", "inter_bra", "state_n")
data = data.drop_nulls().filter(pl.len().over("county") == 12)

In [3]:
with (ro.default_converter + pandas2ri.converter).context():
    r_data = ro.conversion.py2rpy(data.to_pandas())
with ro.default_converter.context():
    ro.globalenv["lending"] = r_data

## Fit the same specification

Without normalization, each horizon describes the effect of the
treatment paths that switchers actually followed. The placebo uses
outcomes before their first treatment change. Both implementations
compare switchers with groups that had the same starting treatment and
had not yet changed it. Those comparison groups include counties whose
treatment never changes.

In [4]:
python_fit = did.did_multiplegt(
    data,
    yname="Dl_vloans_b", tname="year", idname="county", dname="inter_bra",
    effects=2, placebo=1, normalized=False,
    xformla="~1", cluster="state_n", boot=False,
)
print(python_fit)

 Intertemporal Treatment Effects

 Average Total Effect:

┌────────┬────────────┬────────────────────────┬──────┬───────────┐
│    ATE │ Std. Error │ [95% Conf. Interval]   │    N │ Switchers │
├────────┼────────────┼────────────────────────┼──────┼───────────┤
│ 0.0118 │     0.0134 │ [ -0.0146,   0.0381]   │ 4654 │      1642 │
└────────┴────────────┴────────────────────────┴──────┴───────────┘


 Treatment Effects by Horizon:

┌─────────┬──────────┬────────────┬────────────────────────┬──────┬───────────┐
│ Horizon │ Estimate │ Std. Error │ [95% Conf. Interval]   │    N │ Switchers │
├─────────┼──────────┼────────────┼────────────────────────┼──────┼───────────┤
│       1 │   0.0283 │     0.0311 │ [ -0.0326,   0.0891]   │ 3504 │       821 │
│       2 │   0.0251 │     0.0379 │ [ -0.0491,   0.0993]   │ 2753 │       821 │
└─────────┴──────────┴────────────┴────────────────────────┴──────┴───────────┘


 Placebo Effects (Pre-treatment):

┌─────────┬──────────┬────────────┬────────────────

In [5]:
with ro.default_converter.context():
    r_fit = ro.r("""
        DIDmultiplegtDYN::did_multiplegt_dyn(
            df = lending,
            outcome = "Dl_vloans_b", time = "year", group = "county", treatment = "inter_bra",
            effects = 2, placebo = 1, normalized = FALSE,
            controls = NULL, cluster = "state_n", graph_off = TRUE
        )
    """)
    ro.r["print"](r_fit.rx2("results").rx2("Effects"))
    ro.r["print"](r_fit.rx2("results").rx2("Placebos"))

               Estimate         SE       LB CI      UB CI    N Switchers  N.w
Effect_1     0.02827820 0.03105002 -0.03257872 0.08913512 3504       821 3504
Effect_2     0.02510214 0.03785827 -0.04909871 0.09930298 2753       821 2753
             Switchers.w
Effect_1             821
Effect_2             821
               Estimate        SE       LB CI     UB CI    N Switchers  N.w
Placebo_1    0.04976817 0.0485715 -0.04543023 0.1449666 2576       820 2576
             Switchers.w
Placebo_1            820


## Compare estimates and uncertainty

The next table takes each estimate and standard error from its fitted
object. The differences subtract R from Python before any rounding.
Unlike a bootstrap comparison, these analytical standard errors have no
simulation draws to align.

In [6]:
with ro.default_converter.context():
    r_results = r_fit.rx2("results")
    r_effects = r_results.rx2("Effects")
    r_placebos = r_results.rx2("Placebos")

python_estimates = np.r_[python_fit.effects.estimates, python_fit.placebos.estimates]
python_se = np.r_[python_fit.effects.std_errors, python_fit.placebos.std_errors]
with ro.default_converter.context():
    r_estimates = np.r_[r_effects.rx(True, "Estimate"), r_placebos.rx(True, "Estimate")]
    r_se = np.r_[r_effects.rx(True, "SE"), r_placebos.rx(True, "SE")]

comparison = pd.DataFrame({
    "Quantity": ["Effect 1", "Effect 2", "Placebo 1"],
    "Python estimate": python_estimates, "R estimate": r_estimates,
    "Estimate difference": python_estimates - r_estimates,
    "Python SE": python_se, "R SE": r_se,
    "SE difference": python_se - r_se,
})
comparison

,Quantity,Python estimate,R estimate,Estimate difference,Python SE,R SE,SE difference
0,Effect 1,0.02827820,0.02827820,0.00000000e+00,0.03105002,0.03105002,0.0
1,Effect 2,0.02510214,0.02510214,0.00000000e+00,0.03785827,0.03785827,0.0
2,Placebo 1,0.04976817,0.04976817,-6.93889390e-18,0.04857150,0.04857150,0.0


In [7]:
np.testing.assert_allclose(python_estimates, r_estimates, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(python_se, r_se, rtol=1e-3, atol=1e-6)
print(f"Largest estimate difference: {np.max(np.abs(python_estimates - r_estimates)):.2e}")
print(f"Largest standard-error difference: {np.max(np.abs(python_se - r_se)):.2e}")

Largest estimate difference: 6.94e-18
Largest standard-error difference: 0.00e+00


This comparison covers the two horizon estimates, the placebo, and their
state-clustered analytical standard errors. Changing normalization,
controls, or the comparison groups would define a different check.

## Versions used

These versions of Python, R, and the packages produced the output shown on
this page.

In [8]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = str(ro.r("R.version.string")[0])
    did_version = str(ro.r('as.character(packageVersion("DIDmultiplegtDYN"))')[0])
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "DIDmultiplegtDYN"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, did_version],
})

,component,version
0,Python,3.12.10
1,ModernDiD,0.2.0
2,rpy2,3.6.4
3,R,R version 4.3.3 (2024-02-29)
4,DIDmultiplegtDYN,2.4.0
